### <b>Asset definition</b>

In [1]:
from helpers.download import *
from helpers.optimization import *
import pandas as pd

In [11]:


ticker_universe = {
    # US equities — broad & factor
    "SPY":  "S&P 500",
    "QQQ":  "Nasdaq 100 (growth/tech)",
    "IWM":  "Russell 2000 (small-cap blend)",
    "AVUV": "Avantis US Small-Cap Value (small-value factor)",
    "MTUM": "iShares MSCI USA Momentum",
    "USMV": "iShares MSCI USA Min Vol",
    
    "VT":  "Global All-Cap (total world market)",

    # International equities
    "VEA":  "Developed Markets ex-US",
    "VWO":  "Emerging Markets",

    # Bonds — duration spectrum
    "VGSH": "Short-Term Treasury (1-3y)",
    "IEF":  "Intermediate Treasury (7-10y)",
    "VGLT": "Long-Term Treasury (20-30y)",
    "TIP":  "TIPS (inflation-linked)",
    "AGG":  "US Aggregate Bond",
    "HYG":  "High Yield Corporate",

    # Commodities & real assets
    "GLD":  "Gold",
    "SLV":  "Silver",
    "GSG":  "Broad Commodities (S&P GSCI)",
    "VNQ":  "US REITs",
}

interest_tickers = [
    "SPY", "QQQ", "VT", "VEA", "VWO", "VGSH", "IEF", "VGLT", "AGG", "GLD"
]

#download data for the tickers in the universe
start_date, end_date = common_window(interest_tickers)
today = pd.Timestamp.today().strftime("%Y-%m-%d")

print(start_date, end_date)

LOOKBACK_YEARS = 3
REBALANCE_FREQ = "MS"

2010-01-04 00:00:00 2026-07-02 00:00:00


In [ ]:
def build_portfolios(tickers, start, end, lookback_years=LOOKBACK_YEARS, freq=REBALANCE_FREQ):
    
    start = pd.Timestamp(start) + pd.DateOffset(years=lookback_years)
    rebalance_dates = pd.date_range(start, end, freq=freq)

    mv, ms, mc = {}, {}, {}
    for dt in rebalance_dates:
        as_of = dt.date()
        mv[dt] = min_variance(tickers, as_of=as_of, timeframe_years=lookback_years)
        ms[dt] = max_sharpe(tickers, as_of=as_of, timeframe_years=lookback_years)
        # market-cap proxy: 100% VT
        mc[dt] = pd.Series({t: 1.0 if t == "VT" else 0.0 for t in tickers})

    # forward-fill rebalance weights to every business day
    daily_idx = pd.bdate_range(start, end)
    def to_daily(d):
        return pd.DataFrame(d).T.reindex(daily_idx, method="ffill")

    return {
        "min_variance": to_daily(mv),
        "max_sharpe":   to_daily(ms),
        "market_cap":   to_daily(mc),
    }

In [ ]:
portfolios = build_portfolios(interest_tickers, start_date, end_date)

# portfolios["min_variance"], portfolios["max_sharpe"], portfolios["market_cap"]
# each is a DataFrame: index=business days, columns=tickers, values=weights
portfolios["min_variance"].tail()

$VT: possibly delisted; no price data found  (1d 2004-02-01 -> 2008-06-26) (Yahoo error = "Data doesn't exist for startDate = 1075611600, endDate = 1214452800")

1 Failed download:
['VT']: possibly delisted; no price data found  (1d 2004-02-01 -> 2008-06-26) (Yahoo error = "Data doesn't exist for startDate = 1075611600, endDate = 1214452800")
$VEA: possibly delisted; no price data found  (1d 2004-02-01 -> 2007-07-26) (Yahoo error = "Data doesn't exist for startDate = 1075611600, endDate = 1185422400")

1 Failed download:
['VEA']: possibly delisted; no price data found  (1d 2004-02-01 -> 2007-07-26) (Yahoo error = "Data doesn't exist for startDate = 1075611600, endDate = 1185422400")
$VGSH: possibly delisted; no price data found  (1d 2004-02-01 -> 2009-11-23) (Yahoo error = "Data doesn't exist for startDate = 1075611600, endDate = 1258952400")

1 Failed download:
['VGSH']: possibly delisted; no price data found  (1d 2004-02-01 -> 2009-11-23) (Yahoo error = "Data doesn't exist for startD